# **Task 2: Spatial Visualization**

In [ ]:
import os
import sys
import json
from pathlib import Path


import pandas as pd
import numpy as np
import altair as alt


import geopandas as gpd
from shapely.geometry import Point

# Show all rows
pd.set_option('display.max_rows', None)

# Show all columns
pd.set_option('display.max_columns', None)
#pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
#pd.set_option("display.max_colwidth", 200)

print("Versions ->",
      "pandas:", pd.__version__,
      "| geopandas:", gpd.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
file_path = '/content/drive/MyDrive/CS424-Assignment3/df_sub.pkl'
df = pd.read_pickle(file_path)



In [ ]:
#  Building a GeoDataFrame from Longitude/Latitude & sanity-check

# Ensure required columns exist
assert {"Longitude", "Latitude"}.issubset(df.columns), "Longitude/Latitude missing in df_ready."

# Create geometry in WGS84 (EPSG:4326)
gdf_ready = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326"
)

print("GeoDataFrame created.")
print(f"Shape: {gdf_ready.shape}")
print(f"CRS:   {gdf_ready.crs}")

# Spatial bounds check (NYC-ish bbox expected)
minx, miny, maxx, maxy = gdf_ready.total_bounds
print("Total bounds (lon/lat):")
print(f" - lon_min: {minx:.6f}, lon_max: {maxx:.6f}")
print(f" - lat_min: {miny:.6f}, lat_max: {maxy:.6f}")

# Quick borough distribution snapshot (top 10)
if "Borough" in gdf_ready.columns:
    print("\nTop borough counts:")
    display(gdf_ready["Borough"].value_counts().head(10).to_frame("count"))

# Peek a few rows with geometry
gdf_ready[["Borough", "Job Type", "Filing Status", "Latitude", "Longitude", "geometry"]].head(3)


In [ ]:
# to check that GeoDataFrame is present & in sync with FINAL_DF (single step) ---
FINAL_DF = df.copy()
# 1) Create FINAL_GDF if it doesn't exist yet
if 'FINAL_GDF' not in globals():
    assert {'Longitude', 'Latitude'}.issubset(FINAL_DF.columns), "Missing lon/lat in FINAL_DF."
    FINAL_GDF = gpd.GeoDataFrame(
        FINAL_DF.copy(),
        geometry=gpd.points_from_xy(FINAL_DF['Longitude'], FINAL_DF['Latitude']),
        crs='EPSG:4326'
    )
    created = True
else:
    created = False
    # 2) Light sync: ensure non-geometry columns match dtypes/values
    #    (geometry is left untouched)
    missing_in_gdf = [c for c in FINAL_DF.columns if c not in FINAL_GDF.columns]
    for c in missing_in_gdf:
        FINAL_GDF[c] = FINAL_DF[c]

    common = [c for c in FINAL_DF.columns if c in FINAL_GDF.columns and c != 'geometry']
    # If dtype differs, overwrite GDF column with DF version to keep types aligned
    for c in common:
        if FINAL_GDF[c].dtype != FINAL_DF[c].dtype:
            FINAL_GDF[c] = FINAL_DF[c]

    # Ensure CRS is set
    if FINAL_GDF.crs is None:
        FINAL_GDF.set_crs('EPSG:4326', inplace=True)

# 3) Report
print("GeoDataFrame status:")
print(" - Created now?" , created)
print(" - Shape (FINAL_DF): ", FINAL_DF.shape)
print(" - Shape (FINAL_GDF):", FINAL_GDF.shape)
print(" - CRS:", FINAL_GDF.crs)

# 4) Quick column parity check
cols_df  = set(FINAL_DF.columns)
cols_gdf = set(FINAL_GDF.columns) - {'geometry'}
print("\nColumns missing in FINAL_GDF:", sorted(cols_df - cols_gdf) if (cols_df - cols_gdf) else "None")
print("Extra non-geometry columns in FINAL_GDF:", sorted(cols_gdf - cols_df) if (cols_gdf - cols_df) else "None")


####  1) Spatial Density HeatMap of NYC HeatMap

In [ ]:
#  NYC DOB Spatial Density


import pandas as pd, numpy as np
import altair as alt
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path


NTA_GEOJSON_PATH = Path("/content/drive/MyDrive/CS424-Assignment3/nyc_nta.geojson")
BIN_STEP_DEG        = 0.008      # ~0.008° ≈ 900 m; increase for coarser grid
USE_LAST_N_MONTHS   = None
SHOW_BOROUGH_FILTER = True
SIMPLIFY_TOL_FT     = 50
TITLE = "NYC DOB Filings — Spatial Density "

LON_MIN, LON_MAX = -74.28, -73.68
LAT_MIN, LAT_MAX =  40.49,  40.95
APPLY_BBOX = True


alt.data_transformers.disable_max_rows()
alt.renderers.enable("default")

assert 'FINAL_DF' in globals(), "FINAL_DF not found. Load your cleaned dataframe first."
assert NTA_GEOJSON_PATH.exists(), f"NTA GeoJSON not found at: {NTA_GEOJSON_PATH.resolve()}"

df = FINAL_DF.copy()
assert {"Latitude","Longitude"}.issubset(df.columns), "FINAL_DF must have Latitude & Longitude."


df = df.dropna(subset=["Latitude","Longitude"])
if APPLY_BBOX:
    df = df[(df["Longitude"].between(LON_MIN, LON_MAX)) & (df["Latitude"].between(LAT_MIN, LAT_MAX))]


if USE_LAST_N_MONTHS is not None and "Filing Date" in df.columns:
    df = df[df["Filing Date"].notna()].copy()
    latest = df["Filing Date"].max()
    if pd.notna(latest):
        month_start = pd.Timestamp(latest.year, latest.month, 1)
        end_of_month = month_start + pd.offsets.MonthEnd(1)
        last_full_month = month_start if latest >= end_of_month else (month_start - pd.DateOffset(months=1))
        start = last_full_month - pd.DateOffset(months=USE_LAST_N_MONTHS - 1)
        df = df[(df["Filing Date"] >= start) & (df["Filing Date"] <= last_full_month)]

# Read polygons & ensure WGS84
nta = gpd.read_file(NTA_GEOJSON_PATH)
if nta.crs is None:
    nta.set_crs(epsg=4326, inplace=True)
elif nta.crs.to_epsg() != 4326:
    nta = nta.to_crs(4326)


if SIMPLIFY_TOL_FT and SIMPLIFY_TOL_FT > 0:
    try:
        tmp = nta.to_crs(2263)  # NY State Plane (ft)
        tmp["geometry"] = tmp.geometry.simplify(SIMPLIFY_TOL_FT, preserve_topology=True)
        nta = tmp.to_crs(4326)
    except Exception:
        pass

# Spatial clip: keep only points inside NTAs
pnts_gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df["Longitude"], df["Latitude"])],
    crs="EPSG:4326"
)

# 'within' ensures points on borders aren't excluded
pnts_in_nta = gpd.sjoin(pnts_gdf, nta[["geometry"]], predicate="within", how="inner").drop(columns=["index_right"])

# Convert back to pandas for Altair
pts = pd.DataFrame(pnts_in_nta.drop(columns="geometry"))

# Keeping light columns for tooltips (optional)
keep_cols = ["Latitude","Longitude","Borough","Job Type","Filing Date"]
pts = pts[[c for c in keep_cols if c in pts.columns]]

if SHOW_BOROUGH_FILTER and "Borough" in pts.columns:
    borough_values = sorted(pts["Borough"].dropna().unique().tolist())
    borough_sel = alt.param(
        name="borough_sel",
        bind=alt.binding_select(options=["All"] + borough_values, name="Borough: "),
        value="All"
    )
    borough_filter_expr = "(borough_sel == 'All') || (datum.Borough == borough_sel)"
else:
    borough_sel = None
    borough_filter_expr = "true"


heat = (
    alt.Chart(pts)
      .transform_filter(borough_filter_expr)
      .mark_rect()
      .encode(
          x=alt.X("Longitude:Q", bin=alt.Bin(step=BIN_STEP_DEG), title=""),
          y=alt.Y("Latitude:Q",  bin=alt.Bin(step=BIN_STEP_DEG), title=""),
          color=alt.Color("count():Q",
                          title="Filings (sqrt scale)",
                          scale=alt.Scale(type="sqrt")),
          tooltip=[
              alt.Tooltip("count():Q", title="Filings"),
              alt.Tooltip("x:Q", title="Lon bin start", format=".4f"),
              alt.Tooltip("y:Q", title="Lat bin start", format=".4f"),
          ],
      )
      .properties(width=1000, height=600, title=TITLE)
      .project("mercator")
)

if borough_sel is not None:
    heat = heat.add_params(borough_sel)

# -------------------- NTA borders overlay --------------------
nta_json = nta.to_json()
nta_borders = (
    alt.Chart(alt.Data(values=nta_json, format={"type":"json","property":"features"}))
      .mark_geoshape(fill=None, stroke="white", strokeWidth=0.8)
      .project("mercator")
      .properties(width=1000, height=600)
)


final_map = heat + nta_borders
final_map


#### 2) Choropleth of Filings by NTA with Borough Filter

In [ ]:
# --- Choropleth of filings by NTA (with Borough filter) ---

from pathlib import Path
import altair as alt
import geopandas as gpd

alt.data_transformers.disable_max_rows()

NTA_GEOJSON_PATH = Path("/content/drive/MyDrive/CS424-Assignment3/nyc_nta.geojson")

assert 'FINAL_GDF' in globals(), "FINAL_GDF not found. Run the cell that creates/syncs the GeoDataFrame first."
assert NTA_GEOJSON_PATH.exists(), f"NTA GeoJSON not found at: {NTA_GEOJSON_PATH.resolve()}"

# 1) Aggregate unique filings by NTA
nta_counts = (
    FINAL_GDF[['NTA', 'Job Filing Number']]
    .dropna(subset=['NTA'])
    .assign(NTA_key=lambda d: d['NTA'].astype('string').str.strip().str.upper())
    .groupby('NTA_key', observed=False)['Job Filing Number']
    .nunique()
    .rename('filings')
    .reset_index()
)

# 2) Load polygons; detect NTA key and BOROUGH field
gpoly = gpd.read_file(NTA_GEOJSON_PATH).copy()

nta_key_candidates = [
    'NTAName','ntaName','ntaname','NTA','nta',
    'NTA2020','NTA2020_NAME','NTAName20',
    'NTACode','NTACode20','NTACode_20','ntacode','ntacode20'
]
poly_key_col = next((c for c in nta_key_candidates if c in gpoly.columns), None)
if poly_key_col is None:
    raise ValueError(f"Could not find an NTA name/code column in polygons. Available: {list(gpoly.columns)}")

boro_candidates = [
    'BoroName','BORONAME','boro_name','Borough','borough','BOROUGH',
    'BoroCode','BORO','boro'
]
boro_col = next((c for c in boro_candidates if c in gpoly.columns), None)
if boro_col is None:
    raise ValueError(
        "Could not find a borough column in the NTA file. "
        "Add/rename a column to one of: BoroName, BORONAME, borough, Borough, BOROUGH, BoroCode, BORO, boro."
    )

# Build join key and merge counts
gpoly['NTA_key'] = gpoly[poly_key_col].astype('string').str.strip().str.upper()
gpoly = gpoly.merge(nta_counts, how='left', on='NTA_key')
gpoly['filings'] = gpoly['filings'].fillna(0)

# Ensure WGS84
if gpoly.crs is None or gpoly.crs.to_epsg() != 4326:
    gpoly = gpoly.to_crs(epsg=4326)

# 3) Prepare GeoJSON and borough dropdown
gjson = gpoly.to_json()
borough_options = ['All'] + sorted(gpoly[boro_col].astype(str).dropna().unique().tolist())

borough_sel = alt.param(
    name="borough_sel",
    bind=alt.binding_select(options=borough_options, name="Borough: "),
    value="All"
)

# 4) Choropleth with filter
choropleth = (
    alt.Chart(alt.Data(values=gjson, format={'type':'json','property':'features'}))
      .add_params(borough_sel)
      .transform_filter(
          f"(borough_sel == 'All') || (datum.properties['{boro_col}'] == borough_sel)"
      )
      .mark_geoshape(stroke='black', strokeWidth=0.3)
      .encode(
          color=alt.Color('properties.filings:Q',
                          title='Filings (log scale)',
                          scale=alt.Scale(type='symlog', scheme='inferno')),
          tooltip=[
              alt.Tooltip(f'properties["{boro_col}"]:N', title='Borough'),
              alt.Tooltip(f'properties["{poly_key_col}"]:N', title='NTA'),
              alt.Tooltip('properties.filings:Q', title='Filings', format=',.0f')
          ]
      )
      .properties(
          title='Filings by NTA (2021–2025) — with Borough Filter',
          width=900,
          height=600
      )
)

choropleth


#### 3) Choropleth - Scatter Plot with  Metric dropdown (Cost vs Floor Area), Top X% slider

In [ ]:
# Largest Projects — Tiny-Dot Map (metric-colored with P99 clamp) ===
# Controls: Metric dropdown (Cost vs Floor Area), Top X% slider


import pandas as pd, numpy as np
import altair as alt
import geopandas as gpd
from pathlib import Path

alt.data_transformers.disable_max_rows()
alt.renderers.enable("default")

# ---------------- Paths / parameters ----------------
NTA_GEOJSON_PATH = Path("/content/drive/MyDrive/CS424-Assignment3/nyc_nta.geojson")
POINT_SIZE_MIN, POINT_SIZE_MAX = 10, 26     # tiny dots with a slight size accent
TOP_PCT_DEFAULT = 10                        # show top 10% largest projects
TITLE = "Largest Projects — Dot Map"

# ---------------- Checks ----------------
assert 'FINAL_DF' in globals(), "FINAL_DF not found."
assert NTA_GEOJSON_PATH.exists(), f"NTA GeoJSON not found at: {NTA_GEOJSON_PATH.resolve()}"

# ---------------- Prep data ----------------
DF = FINAL_DF.copy()
for c in ["Initial Cost", "Total Construction Floor Area", "Latitude", "Longitude", "Borough"]:
    if c in DF.columns:
        DF[c] = pd.to_numeric(DF[c], errors="coerce") if c not in ["Borough"] else DF[c]

pts = DF.dropna(subset=["Latitude","Longitude"]).copy()

# Optional NYC bbox guard (avoid stray points)
LON_MIN, LON_MAX = -74.28, -73.68
LAT_MIN, LAT_MAX =  40.49,  40.95
pts = pts[(pts["Longitude"].between(LON_MIN, LON_MAX)) & (pts["Latitude"].between(LAT_MIN, LAT_MAX))]

# Build metric columns and percentiles (for filtering)
pts["metric_cost"] = pd.to_numeric(pts.get("Initial Cost"), errors="coerce")
pts["metric_area"] = pd.to_numeric(pts.get("Total Construction Floor Area"), errors="coerce")

def pct_rank_col(s):
    s = s.replace([np.inf,-np.inf], np.nan)
    return (s.rank(pct=True, method="average") * 100.0)

pts["pct_cost"] = pct_rank_col(pts["metric_cost"])
pts["pct_area"] = pct_rank_col(pts["metric_area"])

# Precompute 99th percentile caps for coloring
q99_cost = float(pts["metric_cost"].quantile(0.99))
q99_area = float(pts["metric_area"].quantile(0.99))

# Keep lean columns
keep = ["Latitude","Longitude","Job Filing Number","Borough",
        "metric_cost","metric_area","pct_cost","pct_area"]
keep = [c for c in keep if c in pts.columns]
pts = pts[keep].copy()

# ---------------- Controls ----------------
metric_options = ["Initial Cost", "Total Construction Floor Area"]
metric_sel = alt.param(
    name="metric_sel",
    bind=alt.binding_select(options=metric_options, name="Metric: "),
    value="Initial Cost"
)

top_pct = alt.param(
    name="top_pct",
    bind=alt.binding_range(min=1, max=50, step=1, name="Show top X%: "),
    value=TOP_PCT_DEFAULT
)

# ---------------- Borders (light geoshapes) ----------------
gpoly = gpd.read_file(NTA_GEOJSON_PATH).copy()
if gpoly.crs is None or gpoly.crs.to_epsg() != 4326:
    gpoly = gpoly.to_crs(4326)
nta_json = gpoly.to_json()

borders = (
    alt.Chart(alt.Data(values=nta_json, format={"type":"json","property":"features"}))
      .mark_geoshape(fill=None, stroke="#cccccc", strokeWidth=0.5)
      .project("mercator")
      .properties(width=1100, height=650)
)

# ---------------- Dot layer (color by clamped metric; slight size accent) ----------------
dots = (
    alt.Chart(pts)
      .add_params(metric_sel, top_pct)
      .transform_calculate(
          # raw metric and percentile selector
          metric_raw = "metric_sel == 'Initial Cost' ? datum.metric_cost : datum.metric_area",
          pct        = "metric_sel == 'Initial Cost' ? datum.pct_cost   : datum.pct_area",
          # 99th percentile caps (baked in as constants for speed)
          cap_cost   = f"{q99_cost}",
          cap_area   = f"{q99_area}",
          cap        = "metric_sel == 'Initial Cost' ? datum.cap_cost : datum.cap_area",
          metric     = "datum.metric_raw <= datum.cap ? datum.metric_raw : datum.cap"
      )
      # keep only top X% by chosen metric
      .transform_filter("datum.pct >= 100 - top_pct")
      .mark_circle(opacity=0.9)
      .encode(
          longitude="Longitude:Q",
          latitude="Latitude:Q",
          size=alt.Size(
              "metric:Q",
              title="Metric (clamped at p99)",
              scale=alt.Scale(type="symlog", range=[POINT_SIZE_MIN, POINT_SIZE_MAX]),
              legend=None
          ),
          color=alt.Color(
              "metric:Q",
              title="Scale (clamped at p99)",
              scale=alt.Scale(type="symlog", scheme="inferno", reverse=True)  # dark = largest
          ),
          tooltip=[
              alt.Tooltip("Job Filing Number:N", title="Filing"),
              alt.Tooltip("Borough:N", title="Borough") if "Borough" in pts.columns else alt.value(None),
              alt.Tooltip("metric:Q", title="Metric (clamped)", format=",.0f"),
              alt.Tooltip("pct:Q", title="Percentile", format=".1f")
          ]
      )
      .project("mercator")
      .properties(title=TITLE, width=1100, height=650)
)

final = (borders + dots).configure_view(stroke=None).configure(background="white")
final


#### 4) Choropleth - Dwelling Unit Impact Filtered by boroughs and minimum X%

In [ ]:
# Dwelling-Unit Impact (ΔDU) by NTA — Diverging Choropleth
# ΔDU = Proposed Dwelling Units − Existing Dwelling Units


import pandas as pd, numpy as np
import altair as alt
import geopandas as gpd
from pathlib import Path

alt.data_transformers.disable_max_rows()
alt.renderers.enable("default")

# ---------- Paths / params ----------
NTA_GEOJSON_PATH = Path("/content/drive/MyDrive/CS424-Assignment3/nyc_nta.geojson")

COLOR_SCHEME   = "redblue"  # good diverging option; you can try "blueorange", "redgrey", etc.
MIN_N_DEFAULT  = 20         # hide areas with too few filings to reduce noisy ΔDU
USE_MEDIAN_IN_COLOR = False # if True, color by median ΔDU; otherwise by sum ΔDU

# ---------- Validate ----------
assert 'FINAL_DF' in globals(), "FINAL_DF not found."
assert NTA_GEOJSON_PATH.exists(), f"NTA GeoJSON not found at: {NTA_GEOJSON_PATH.resolve()}"

# ---------- Compute ΔDU per filing ----------
DF = FINAL_DF.copy()

for c in ["Existing Dwelling Units","Proposed Dwelling Units"]:
    DF[c] = pd.to_numeric(DF[c], errors="coerce")

DF["ΔDU"] = DF["Proposed Dwelling Units"] - DF["Existing Dwelling Units"]

# keep rows with an NTA and finite ΔDU
DF = DF.dropna(subset=["NTA","ΔDU"]).copy()

# ---------- Aggregate by NTA ----------
agg = (DF
       .assign(NTA_key=lambda d: d["NTA"].astype("string").str.strip().str.upper())
       .groupby("NTA_key", observed=False)
       .agg(
           delta_sum   = ("ΔDU","sum"),
           delta_median= ("ΔDU","median"),
           n           = ("ΔDU","size")
       )
       .reset_index())

# ---------- Load polygons & detect join keys ----------
gpoly = gpd.read_file(NTA_GEOJSON_PATH).copy()

nta_key_candidates = [
    "NTAName","ntaName","ntaname","NTA","nta",
    "NTA2020","NTA2020_NAME","NTAName20",
    "NTACode","NTACode20","NTACode_20","ntacode","ntacode20"
]
poly_key_col = next((c for c in nta_key_candidates if c in gpoly.columns), None)
if poly_key_col is None:
    raise ValueError(f"Could not find an NTA name/code column in polygons. Available: {list(gpoly.columns)}")

boro_candidates = ["BoroName","BORONAME","boro_name","Borough","borough","BOROUGH","BoroCode","BORO","boro"]
boro_col = next((c for c in boro_candidates if c in gpoly.columns), None)  # optional

# Join key, CRS
gpoly["NTA_key"] = gpoly[poly_key_col].astype("string").str.strip().str.upper()
if gpoly.crs is None or gpoly.crs.to_epsg() != 4326:
    gpoly = gpoly.to_crs(4326)

# Merge stats
gpoly = gpoly.merge(agg, how="left", on="NTA_key")
gpoly[["delta_sum","delta_median","n"]] = gpoly[["delta_sum","delta_median","n"]].fillna(0)

# ---------- Prepare GeoJSON for Altair ----------
gjson = gpoly.to_json()

# Controls
controls = []
min_n_slider = alt.param(
    name="min_n",
    value=MIN_N_DEFAULT,
    bind=alt.binding_range(min=0, max=int(max(1, gpoly["n"].max())), step=1, name="Min filings (n): ")
)
controls.append(min_n_slider)

borough_filter_expr = "true"
if boro_col is not None:
    borough_options = ["All"] + sorted(gpoly[boro_col].astype(str).dropna().unique().tolist())
    borough_sel = alt.param(
        name="borough_sel",
        bind=alt.binding_select(options=borough_options, name="Borough: "),
        value="All"
    )
    controls.append(borough_sel)
    borough_filter_expr = f"(borough_sel == 'All') || (datum.properties['{boro_col}'] == borough_sel)"

# Which metric drives color?
metric_field = "properties.delta_median:Q" if USE_MEDIAN_IN_COLOR else "properties.delta_sum:Q"
metric_title = "ΔDU (median)" if USE_MEDIAN_IN_COLOR else "ΔDU (sum)"

# Build chart
choropleth = (
    alt.Chart(alt.Data(values=gjson, format={"type":"json","property":"features"}))
      .add_params(*controls)
      .transform_filter(borough_filter_expr)
      .transform_filter("datum.properties.n >= min_n")  # hide small-n areas
      .mark_geoshape(stroke="white", strokeWidth=0.35)
      .encode(
          color=alt.Color(
              metric_field,
              title=metric_title,
              # Diverging around 0; symlog handles skew (big gains) but keeps 0 meaningful
              scale=alt.Scale(type="symlog", domainMid=0, scheme=COLOR_SCHEME)
          ),
          tooltip=[
              alt.Tooltip(f'properties["{poly_key_col}"]:N', title="NTA"),
              alt.Tooltip('properties.n:Q',               title="Filings (n)", format=",d"),
              alt.Tooltip('properties.delta_sum:Q',       title="ΔDU sum",     format=",d"),
              alt.Tooltip('properties.delta_median:Q',    title="ΔDU median",  format=".1f"),
          ]
      )
      .properties(
          title="Dwelling-Unit Impact (ΔDU) by NTA — Diverging around 0",
          width=900, height=600
      )
      .project("mercator")
)

choropleth


#### 5) Dot Map Choropleth of Largest Projects wrt to Initial Cost and Total Construction Area

In [ ]:
# === Dot Map of Largest Projects (one point per filing) ===


import pandas as pd, numpy as np
import altair as alt
import geopandas as gpd
from pathlib import Path

alt.data_transformers.disable_max_rows()
alt.renderers.enable("default")


NTA_GEOJSON_PATH = Path("/content/drive/MyDrive/CS424-Assignment3/nyc_nta.geojson")
POINT_SIZE       = 12        # fixed size for that “many tiny dots” look
TOP_PCT_DEFAULT  = 10        # show top 10% biggest projects by chosen metric
TITLE = "Largest Projects — Dot Map"

assert 'FINAL_DF' in globals(), "FINAL_DF not found."
assert NTA_GEOJSON_PATH.exists(), f"NTA GeoJSON not found at: {NTA_GEOJSON_PATH.resolve()}"


DF = FINAL_DF.copy()

# numeric
for c in ["Initial Cost", "Total Construction Floor Area", "Latitude", "Longitude"]:
    if c in DF.columns:
        DF[c] = pd.to_numeric(DF[c], errors="coerce")

# keep valid points + metrics
pts = DF.dropna(subset=["Latitude", "Longitude"]).copy()


def pct_rank(s):
    s = s.replace([np.inf, -np.inf], np.nan)
    return s.rank(pct=True, method="average")

pts["metric_cost"] = pts["Initial Cost"]
pts["metric_area"] = pts["Total Construction Floor Area"]

pts["pct_cost"] = pct_rank(pts["metric_cost"]) * 100.0
pts["pct_area"] = pct_rank(pts["metric_area"]) * 100.0


LON_MIN, LON_MAX = -74.28, -73.68
LAT_MIN, LAT_MAX =  40.49,  40.95
pts = pts[(pts["Longitude"].between(LON_MIN, LON_MAX)) & (pts["Latitude"].between(LAT_MIN, LAT_MAX))]

keep = ["Latitude","Longitude","Job Filing Number","Borough","metric_cost","metric_area","pct_cost","pct_area"]
keep = [c for c in keep if c in pts.columns]
pts = pts[keep].copy()


metric_options = ["Initial Cost", "Total Construction Floor Area"]
metric_sel = alt.param(
    name="metric_sel",
    bind=alt.binding_select(options=metric_options, name="Metric: "),
    value="Initial Cost"
)

top_pct = alt.param(
    name="top_pct",
    bind=alt.binding_range(min=1, max=50, step=1, name="Show top X%: "),
    value=TOP_PCT_DEFAULT
)


gpoly = gpd.read_file(NTA_GEOJSON_PATH).copy()
if gpoly.crs is None or gpoly.crs.to_epsg() != 4326:
    gpoly = gpoly.to_crs(4326)
nta_json = gpoly.to_json()

borders = (
    alt.Chart(alt.Data(values=nta_json, format={"type":"json","property":"features"}))
      .mark_geoshape(fill=None, stroke="lightgray", strokeWidth=0.6)
      .project("mercator")
      .properties(width=1000, height=630)
)


dots = (
    alt.Chart(pts)
      .add_params(metric_sel, top_pct)
      .transform_calculate(
          # switch metric & its percentile based on dropdown
          pct   ="metric_sel == 'Initial Cost' ? datum.pct_cost   : datum.pct_area"
      )
      .transform_filter("datum.pct >= 100 - top_pct")  # already selecting top X%
      .mark_circle(size=12, opacity=0.75)
      .encode(
          longitude="Longitude:Q",
          latitude="Latitude:Q",
          color=alt.Color(
              "pct:Q",
              title="Percentile (citywide)",
              scale=alt.Scale(domain=[90, 100], scheme="inferno", reverse=True)  # 🔥 dark=highest
          ),
          tooltip=[
              alt.Tooltip("Job Filing Number:N", title="Filing"),
              alt.Tooltip("Borough:N", title="Borough") if "Borough" in pts.columns else alt.value(None),
              alt.Tooltip("pct:Q", title="Percentile", format=".1f")
          ]
      )
      .project("mercator")
      .properties(title=TITLE + " — colored by percentile", width=1000, height=630)
)

borders + dots


### FINAL CHOROPLETH : Mapping DOB Filings (also wrt to Job Types) with param bubbles, brush,  and linked time series

In [ ]:
# --- Choropleth + param-bubbles + linked time series ---



alt.data_transformers.disable_max_rows()

# -------------------- INPUTS --------------------

NTA_GEOJSON_PATH = Path("/content/drive/MyDrive/CS424-Assignment3/nyc_nta.geojson")   # your NTA polygons



# Ensure we have Filing Date + NTA + Job Type + unique filing id
pts = (
    FINAL_GDF[["Filing Date", "NTA", "Job Type", "Job Filing Number"]]
    .dropna(subset=["Filing Date", "NTA", "Job Type"])
    .copy()
)
pts["NTA_key"] = pts["NTA"].astype('string').str.strip().str.upper()
pts["month_start"] = pts["Filing Date"].values.astype("datetime64[M]")
pts["mmYYYY"] = pts["month_start"].dt.strftime("%Y-%m")

# -------------------- POLYGONS & STATIC CHOROPLETH DATA --------------------
gpoly = gpd.read_file(NTA_GEOJSON_PATH)

candidate_cols = [
    'NTAName','ntaName','ntaname',
    'NTA','nta',
    'NTA2020','NTA2020_NAME','NTAName20',
    'NTACode','NTACode20','NTACode_20',
    'ntacode','ntacode20'
]
poly_key_col = next((c for c in candidate_cols if c in gpoly.columns), None)
if poly_key_col is None:
    raise ValueError(
        f"Could not find an NTA name/code column in polygons. "
        f"Available columns: {list(gpoly.columns)}\n"
        f"Tip: rename the proper column to 'NTAName' and re-run."
    )

gpoly = gpoly.copy()
gpoly["NTA_key"] = gpoly[poly_key_col].astype("string").str.strip().str.upper()
gpoly["NTA_label"] = gpoly[poly_key_col].astype("string")

# Static 2021–2025 total filings per NTA for base choropleth color
nta_totals = (
    pts.assign(yr=pts["month_start"].dt.year)
       .query("yr >= 2021 and yr <= 2025")
       .groupby("NTA_key", observed=False)["Job Filing Number"]
       .nunique()
       .rename("filings_total")
       .reset_index()
)

# Merge totals into polygons
gpoly2 = gpoly.merge(nta_totals, on="NTA_key", how="left")
gpoly2["filings_total"] = gpoly2["filings_total"].fillna(0)

# Ensure WGS84
if gpoly2.crs is None or gpoly2.crs.to_epsg() != 4326:
    gpoly2 = gpoly2.to_crs(epsg=4326)

# Robust centroids: compute in projected CRS, then back to 4326
gpoly_proj = gpoly2.to_crs(epsg=2263)  # NAD83 / New York Long Island (ftUS)
cent_geom_proj = gpoly_proj.geometry.centroid
cent_geom_wgs = gpd.GeoSeries(cent_geom_proj, crs=2263).to_crs(epsg=4326)
centroids_df = pd.DataFrame({
    "NTA_key": gpoly2["NTA_key"].values,
    "lon": cent_geom_wgs.x.values,
    "lat": cent_geom_wgs.y.values
})


nta_monthly = (
    pts.groupby(["NTA_key","Job Type","month_start","mmYYYY"], observed=False)["Job Filing Number"]
       .nunique()
       .rename("filings")
       .reset_index()
)

city_monthly = (
    pts.groupby(["month_start","mmYYYY","Job Type"], observed=False)["Job Filing Number"]
       .nunique()
       .rename("filings_city")
       .reset_index()
)

jobtype_opts = sorted(pts["Job Type"].dropna().astype(str).unique().tolist())

month_order = (
    city_monthly[["month_start","mmYYYY"]]
    .drop_duplicates()
    .sort_values("month_start")["mmYYYY"]
    .tolist()
)

# -------------------- INTERACTIVE PARAMS --------------------
p_job = alt.param(
    name="jobSel", value="(All)",
    bind=alt.binding_select(options=["(All)"] + jobtype_opts, name="Job Type")
)
brush = alt.selection_interval(encodings=["x"])
nta_click = alt.selection_point(fields=["NTA_key"])

# -------------------- CHOROPLETH (symlog only) --------------------
gjson = gpoly2.to_json()

choropleth = (
    alt.Chart(alt.Data(values=gjson, format={'type': 'json', 'property': 'features'}))
      .transform_calculate(NTA_key="datum.properties.NTA_key")
      .mark_geoshape(stroke='black', strokeWidth=0.25)
      .encode(
          color=alt.Color(
              'properties.filings_total:Q',
              title='Total filings 2021–2025',
              scale=alt.Scale(type='symlog', scheme='viridis')
          ),
          tooltip=[
              alt.Tooltip(f'properties.{poly_key_col}:N', title='NTA'),
              alt.Tooltip('properties.filings_total:Q', title='Total filings', format=','),
          ],
          opacity=alt.condition(nta_click, alt.value(0.95), alt.value(0.75))
      )
      .add_params(nta_click)
      .properties(width=520, height=620, title="Filings by NTA (symlog)")
)

# -------------------- BUBBLES (counts within brushed window) --------------------
bubbles = (
    alt.Chart(nta_monthly)
      .transform_calculate(passJob="(isValid(jobSel) && jobSel != '(All)') ? datum['Job Type'] == jobSel : true")
      .transform_filter("datum.passJob")
      .transform_filter(brush)  # only the brushed months
      .transform_aggregate(total_filings="sum(filings)", groupby=["NTA_key"])
      .transform_lookup(
          lookup="NTA_key",
          from_=alt.LookupData(centroids_df, key="NTA_key", fields=["lon","lat"])
      )
      .mark_circle(stroke="black", strokeWidth=0.5)
      .encode(
          longitude="lon:Q",
          latitude="lat:Q",
          size=alt.Size("total_filings:Q", title="Filings (in brush)", scale=alt.Scale(range=[10, 1200])),
          color=alt.Color("total_filings:Q", title="Filings (in brush)", scale=alt.Scale(scheme="plasma")),
          tooltip=[
              alt.Tooltip("NTA_key:N", title="NTA key"),
              alt.Tooltip("total_filings:Q", title="Filings (window)", format=","),
          ],
          opacity=alt.condition(nta_click, alt.value(1.0), alt.value(0.9))
      )
      .properties(width=520, height=620)
)

map_comp = alt.layer(choropleth, bubbles).resolve_scale(color="independent", size="independent")

# -------------------- LINKED TIME SERIES --------------------
ts_base = alt.Chart(nta_monthly).properties(width=900, height=180)

ts_sel = (
    ts_base
    .transform_filter(nta_click)
    .transform_calculate(passJob="(isValid(jobSel) && jobSel != '(All)') ? datum['Job Type'] == jobSel : true")
    .transform_filter("datum.passJob")
    .transform_aggregate(filings="sum(filings)", groupby=["mmYYYY","month_start"])
    .mark_line(point=True)
    .encode(
        x=alt.X("mmYYYY:N", sort=month_order, title="Month"),
        y=alt.Y("filings:Q", title="Filings (selected NTA)"),
        tooltip=[alt.Tooltip("mmYYYY:N", title="Month"),
                 alt.Tooltip("filings:Q", title="Filings", format=",")]
    )
)

ts_city = (
    alt.Chart(city_monthly)
    .transform_calculate(passJob="(isValid(jobSel) && jobSel != '(All)') ? datum['Job Type'] == jobSel : true")
    .transform_filter("datum.passJob")
    .transform_aggregate(filings_city="sum(filings_city)", groupby=["mmYYYY","month_start"])
    .mark_area(opacity=0.25)
    .encode(
        x=alt.X("mmYYYY:N", sort=month_order, title="Month (brush to set map window)"),
        y=alt.Y("filings_city:Q", title="Citywide filings"),
        tooltip=[alt.Tooltip("mmYYYY:N", title="Month"),
                 alt.Tooltip("filings_city:Q", title="Citywide", format=",")]
    )
    .add_params(brush)
    .properties(width=900, height=120)
)

ts_title = (
    alt.Chart(pd.DataFrame({'t': ['Monthly filings — select an NTA on map; brush to aggregate bubbles']}))
      .mark_text(dy=-5, fontSize=13, fontWeight='bold', align='left')
      .encode(text='t:N')
      .properties(width=900, height=20)
)

ts_comp = alt.vconcat(ts_title, ts_city, ts_sel).resolve_scale(x="shared")


final_map_dashboard = alt.hconcat(
    map_comp,
    ts_comp
).add_params(p_job).properties(
    title="NTA Filings — Symlog Choropleth + Bubble Overlay (Job-type & Brush Filtered) with Linked Time Series"
)

final_map_dashboard
